In [1]:
import sys, pandas as pd, json
from pathlib import Path

project_root = Path.cwd()

if not (project_root / "src").is_dir():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

from src.load_cloudtrail import load_cloudtrail
from src.cloudtrail_adapter import adapt_cloudtrail_event


print("Project root:", project_root)

Project root: /Users/gerardoramirez/Library/CloudStorage/GoogleDrive-gerardoram1010@gmail.com/My Drive/Portfolio Projects/cloud-security-intelligence-platform


In [2]:
file_path = project_root / "data" / "synthetic_cloudtrail.json"

records = load_cloudtrail(str(file_path))

print("Records loaded: ", len(records))

Records loaded:  100


In [3]:
normalized_events = []

for record in records:
    normalized = adapt_cloudtrail_event(record)
    normalized_events.append(normalized)

Invalid source IP address.
Invalid source IP address.
Invalid source IP address.
Invalid source IP address.
Invalid source IP address.
Invalid source IP address.
Invalid source IP address.
Invalid source IP address.
Invalid source IP address.
Invalid source IP address.


In [4]:
df = pd.DataFrame(normalized_events)
df.head()

,timestamp,user,action,region,resource,security_event,ip_metadata,raw_record
0,2026-11-11T07:00:00+00:00,admin,ConsoleLogin,eu-west-1,analytics-logs,SecurityEvent(timestamp='2026-11-11T07:00:00+0...,"{'source_ip_raw': '198.51.100.5', 'source_ip_t...","{'eventTime': '2026-11-11T07:00:00+00:00', 'ev..."
1,2026-11-10T02:00:00+00:00,admin,PutObject,us-west-2,analytics-logs,SecurityEvent(timestamp='2026-11-10T02:00:00+0...,"{'source_ip_raw': '203.0.113.42', 'source_ip_t...","{'eventTime': '2026-11-10T02:00:00+00:00', 'ev..."
2,2026-11-14T13:00:00+00:00,admin,PutObject,us-east-1,analytics-logs,SecurityEvent(timestamp='2026-11-14T13:00:00+0...,"{'source_ip_raw': '192.0.2.15', 'source_ip_typ...","{'eventTime': '2026-11-14T13:00:00+00:00', 'ev..."
3,2026-11-12T08:00:00+00:00,admin,ConsoleLogin,eu-west-1,analytics-logs,SecurityEvent(timestamp='2026-11-12T08:00:00+0...,"{'source_ip_raw': '198.51.100.5', 'source_ip_t...","{'eventTime': '2026-11-12T08:00:00+00:00', 'ev..."
4,2026-11-10T11:00:00+00:00,developer,ConsoleLogin,us-east-1,analytics-logs,SecurityEvent(timestamp='2026-11-10T11:00:00+0...,"{'source_ip_raw': '198.51.100.5', 'source_ip_t...","{'eventTime': '2026-11-10T11:00:00+00:00', 'ev..."


In [5]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Type:", df.dtypes)

Shape: (100, 8)
Columns: ['timestamp', 'user', 'action', 'region', 'resource', 'security_event', 'ip_metadata', 'raw_record']
Type: timestamp            str
user                 str
action               str
region               str
resource             str
security_event    object
ip_metadata       object
raw_record        object
dtype: object


In [6]:
# Changing timestamp str -> datetime
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

In [7]:
missing_values = df.isna().sum()
print("Number of missing values in dataframe: ", missing_values)

Number of missing values in dataframe:  timestamp          0
user               0
action             0
region             0
resource           0
security_event    10
ip_metadata        0
raw_record         0
dtype: int64


In [8]:
df["source_ip"] = df["ip_metadata"].apply(
    lambda metadata: metadata.get("source_ip")
)

print("Missing source IPs: ", df["source_ip"].isna().sum())

Missing source IPs:  10


In [9]:
matching_missingness = (
    df["source_ip"].isna()
    == df["security_event"].isna()
)

print("All rows match:", matching_missingness.all())

All rows match: True


In [10]:
print("Unique identity count:", df["user"].nunique())

print("Identities:", df["user"].unique())

Unique identity count: 4
Identities: <StringArray>
['admin', 'developer', 'analyst', 'service-user']
Length: 4, dtype: str


In [11]:
# Using JSON seralization to create json strings of the record that can be checked for duplicates

df["raw_record_json"] = df["raw_record"].apply(
    lambda record: json.dumps(record, sort_keys=True)
)

In [12]:
duplicate_count = df["raw_record_json"].duplicated().sum()

print("Duplicate records:", duplicate_count)

Duplicate records: 5


In [13]:
print("Record 0 matches record 95:",
      df.loc[0, "raw_record_json"] == df.loc[95, "raw_record_json"])

Record 0 matches record 95: True


Day 13 — Data Quality Findings

Our dataset contains 100 synthetic AWS CloudTrail-style records representing four unique identities across multiple days.

10 records (10%) have missing source IP addresses. These records are preserved by our adapter rather than discarded, since CloudTrail records may contain missing source IPs or AWS service identifiers.

We identified 5 exact duplicate records (5%), leaving 95 distinct record contents. These duplicates could distort activity counts and failure rates if included in our calculations. Before removing them, we should investigate whether they also contain missing values or failed actions.

All timestamps were successfully converted into timezone-aware UTC datetime values, enabling reliable time-based analysis.

Limitations: Our dataset is artificially generated and does not necessarily reflect real AWS activity. It also lacks unique event identifiers, such as eventID, making it difficult to distinguish genuinely separate events from duplicate observations with identical contents. Therefore, conclusions from this dataset should not be generalized to real-world security activity.